# 06 · Direction 2: training objectives for graded labels

**Question:** with the backbone, input and schedule fixed, does the training objective change the ranking? ESCI labels are graded (E > S > C > I) and the metric weights them 1 / 0.1 / 0.01 / 0. B3 regresses the gain with MSE; this notebook trains the same model with two other objectives.

| Run | Loss | Score used for ranking | Seeds | Compared with |
|---|---|---|---|---|
| `b3_minilm_title_mse` (already run, notebook 03) | MSE on the gain | the output | 42, 43 | – |
| `d2_minilm_title_ce4` | 4-class cross-entropy on E/S/C/I | expected gain p_E + 0.1 p_S + 0.01 p_C | 42, 43 | B3, same seed |
| `d2_minilm_title_listnet` | ListNet: softmax over each query list vs target gain / sum(gain) | the output | 42, 43 | B3, same seed |

**Settings.** Everything except the loss is B3: `cross-encoder/ms-marco-MiniLM-L-12-v2`, `title`, `max_len` 128, 1 epoch, batch 32, lr 7e-6, 5,000 warm-up steps, fp16. MiniLM `title` is the cheapest configuration (~0.2 h per epoch), so it is used for the comparison; the best one or two losses are re-run later on the configuration chosen in Direction 1.

Differences that come with the loss itself:
- **ce4** needs four outputs, so its classification head is new (random), while MSE and ListNet keep the one-output MS MARCO head. Same batches and number of steps as B3.
- **ListNet** needs whole query lists in a batch: each query's candidates are split into lists of at most 16 (near-equal sizes), lists are packed into batches of at most 32 pairs. Every pair is seen once per epoch as in B3, but batches are slightly smaller, so there are more steps (printed below). Lists with only I labels are skipped (no ordering to learn).

**Result on MiniLM `title` (val nDCG, mean of 2 seeds):** MSE 0.8572 · ce4 0.8570 · ListNet 0.8579; no paired difference with B3 is significant for either seed, E-over-S is unchanged. The loss does not matter at this size.

**Stage 2, `SETUP = 'deberta_main'`:** ce4 on the Direction 1 configuration (DeBERTa-v3-base, `main`, `max_len` 256, lr 2e-5, 1,200 warm-up steps, settings of notebook 05), seed 42, compared with `d1_deberta_main_mse_s42`. ce4 is kept because it ties with MSE and also gives p_E / p_S / p_C / p_I, which are features for the LightGBM re-ranker (Direction 3). DeBERTa has no pretrained head, so here MSE and ce4 both start from a new head. If `d1_deberta_main_mse_s42` is not in the inputs yet, the paired comparison is done later in `10_report` from the saved scores.

**Test is blind**, as in Direction 1: test scores are saved, not scored; `10_report` scores them.

**Kaggle settings:** GPU **T4**, **Internet on** (Hugging Face model).

**Inputs:**
- Dataset `wm-src` (zip of `src/` and `data/results/`, see README), version with the `ce4` / `listnet` losses in `src/cross_encoder.py`. Required. Its `data/results/` holds the B3 scores for the paired comparisons.
- Add Input → Notebook Output → **01_preprocess** (`pairs.parquet`, `products_clean.parquet`). Required.

**How to run.** Set `SETUP`, then:
1. `MODE = 'smoke'`: 5% of train, seed 42, val only. `minilm_title`: MSE, ce4 and ListNet (about 10 minutes); `deberta_main`: ce4 only (about 15 minutes). Check the loss self-check in section 0 and the verdict at the end of section 2.
2. `MODE = 'full'`, then **Save Version → Save & Run All** (`minilm_title` about 1 to 1.5 h, done; `deberta_main` about 2.6 h). Download `results/` into `data/results/` in the repo.

**Output** (`/kaggle/working/results/`): `<run>.json` (val metrics, config, loss log, val curve, `src_version`) and `scores/<run>_{val,test}.parquet`.

In [ ]:
# Setup: make src/ importable. On Kaggle, attach the dataset `wm-src` (zip of src/, see README).
import glob, sys
from pathlib import Path

_hits = glob.glob('/kaggle/input/**/src/evaluation.py', recursive=True) + glob.glob('../src/evaluation.py')
assert _hits, 'src/ not found: Add Input -> dataset wm-src (zip of src/, see README)'
ROOT = Path(_hits[0]).resolve().parents[1]
sys.path.insert(0, str(ROOT))

from src.evaluation import src_version
print('src from', ROOT, '|', src_version())

In [ ]:
import gc, os, time

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')   # less fragmentation; before torch

import numpy as np
import pandas as pd
import torch
import transformers

import src.cross_encoder as ce
from src.cross_encoder import predict, train_ce
from src.data import load_pairs
from src.evaluation import RunLog

assert hasattr(ce, 'listnet_loss'), 'wm-src is an old version: rebuild the zip with the D2 losses and update the input'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
transformers.logging.set_verbosity_error()

MODE = 'smoke'           # 'smoke': 5% of train, seed 42, val only | 'full': all train
SETUP = 'deberta_main'   # 'minilm_title' (stage 1, done) | 'deberta_main' (stage 2: ce4 on the Direction 1 config)

BASE = {'epochs': 1, 'batch_size': 32, 'weight_decay': 0.01, 'max_grad_norm': 1.0, 'fp16': True}
SETUPS = {
    # full MSE run of each setup already exists (B3, notebook 05), so it is only re-run in smoke mode for MiniLM
    'minilm_title': {'cfg': {**BASE, 'model': 'cross-encoder/ms-marco-MiniLM-L-12-v2', 'fields': 'title', 'max_len': 128,
                             'lr': 7e-6, 'warmup_steps': 5000},                                       # = B3
                     'losses': ['ce4', 'listnet'], 'smoke_losses': ['mse', 'ce4', 'listnet'], 'seeds': [42, 43],
                     'ref': 'b3_minilm_title_mse_s{seed}', 'smoke_eval_every': 100},
    'deberta_main': {'cfg': {**BASE, 'model': 'microsoft/deberta-v3-base', 'fields': 'main', 'max_len': 256,
                             'lr': 2e-5, 'warmup_steps': 1200, 'eval_batch_size': 64},                 # = notebook 05
                     'losses': ['ce4'], 'smoke_losses': ['ce4'], 'seeds': [42],
                     'ref': 'd1_deberta_main_mse_s{seed}', 'smoke_eval_every': 300},
}
EXTRA = {'listnet': {'list_size': 16}}
S = SETUPS[SETUP]
LOSSES = S['smoke_losses'] if MODE == 'smoke' else S['losses']
SEEDS = [42] if MODE == 'smoke' else S['seeds']
REF = S['ref']

SEED = 42
SUBSAMPLE = 0.05 if MODE == 'smoke' else 1.0
EVAL_SPLITS = ['val'] if MODE == 'smoke' else ['val', 'test']   # test is predicted and saved, never scored here
EVAL_EVERY = S['smoke_eval_every'] if MODE == 'smoke' else 4000  # steps between val checks (monitoring only)
RES = Path('/kaggle/working/results')

assert MODE in ('smoke', 'full')
assert torch.cuda.is_available(), 'No GPU: Settings -> Accelerator -> GPU T4'
print('torch', torch.__version__, '| transformers', transformers.__version__, '|', torch.cuda.get_device_name(0))
print('MODE', MODE, '| SETUP', SETUP, '| losses', LOSSES, '| seeds', SEEDS)

## 0. Loss self-check

Small hand-made cases, on CPU: `listnet_loss` against a direct per-list computation, and `query_batches` covering every row of lists longer than one exactly once.

In [ ]:
rng = np.random.default_rng(0)
pred = torch.randn(9)
gain = torch.tensor([1, 0.1, 0, 0.01, 1, 0, 0, 0, 0.1])
seg = torch.tensor([0, 0, 0, 1, 1, 1, 2, 2, 3])   # list 2 has no gain (skipped); list 3 has one row
naive = []
for s in range(4):
    p, g = pred[seg == s], gain[seg == s]
    if g.sum() > 0:
        naive.append(-(g / g.sum() * torch.log_softmax(p, 0)).sum())
assert torch.allclose(ce.listnet_loss(pred, gain, seg), torch.stack(naive).mean(), atol=1e-6)

qids = np.repeat(np.arange(50), rng.integers(1, 40, 50))
batches = ce.query_batches(qids, np.random.default_rng(1), list_size=16, batch_pairs=32)
rows = np.concatenate([i for i, _ in batches])
sizes = pd.Series(qids).value_counts()
assert len(rows) == len(set(rows)) == sizes[sizes > 1].sum()
for i, s in batches:
    assert len(i) <= 32
    for k in range(s.max() + 1):
        assert len(set(qids[i[s == k]])) == 1 and 2 <= (s == k).sum() <= 16
print('loss self-check passed |', len(batches), 'batches for', len(rows), 'rows')

## 1. Data

Same pairs as every run; product text of the setup (`title` or `main`). In smoke mode, 5% of the training queries.

In [ ]:
pairs = load_pairs(S['cfg']['fields'])
MASK = {s: (pairs['split'] == s).to_numpy() for s in ['val', 'test']}
train_df = pairs[pairs['split'] == 'train']
if SUBSAMPLE < 1:
    keep = train_df['query_id'].drop_duplicates().sample(frac=SUBSAMPLE, random_state=SEED)
    train_df = train_df[train_df['query_id'].isin(keep)]
train_df = train_df.reset_index(drop=True)
parts = {s: pairs[MASK[s]] for s in EVAL_SPLITS}
print(f'train {train_df["query_id"].nunique():,} queries / {len(train_df):,} pairs | '
      f'val {MASK["val"].sum():,} | test {MASK["test"].sum():,} pairs')

## 2. Training runs

Each run trains with `train_ce` and logs val metrics with `RunLog`; test scores are saved blind. The MSE run of the same setup and seed (B3, or `d1_deberta_main_mse`) is loaded from the inputs when present, for the paired bootstrap on val nDCG.

In [ ]:
log = RunLog(RES, {s: parts[s][['example_id', 'query_id', 'esci_label']] for s in EVAL_SPLITS})
if MODE == 'full':
    for s in SEEDS:
        if not log.load(REF.format(seed=s), splits=['val']):
            print(f'{REF.format(seed=s)}: not found in the inputs (no paired comparison)')

runs = []
for loss in LOSSES:
    cfg = {**S['cfg'], 'loss': loss, **EXTRA.get(loss, {})}
    for seed in SEEDS:
        name = f"{'smoke_' if MODE == 'smoke' else ''}d2_{SETUP}_{loss}_s{seed}"
        print(f'\n==== {name}')
        model, tokenizer, info = train_ce(cfg, train_df, seed, eval_df=parts['val'], eval_every=EVAL_EVERY,
                                          print_every=100 if MODE == 'smoke' else 1000)
        t0 = time.time()
        torch.cuda.empty_cache()
        scores = {s: predict(model, tokenizer, parts[s], cfg['max_len'], batch_size=cfg.get('eval_batch_size', 512),
                             fp16=cfg['fp16']) for s in EVAL_SPLITS}
        log.log(name, scores, {'mode': MODE, **cfg, 'seed': seed, 'subsample': SUBSAMPLE,
                               'train_queries': int(train_df['query_id'].nunique()), 'train_pairs': len(train_df),
                               'gpu': torch.cuda.get_device_name(0), 'predict_minutes': round((time.time() - t0) / 60, 1),
                               **info},
                compare_to=REF.format(seed=seed) if MODE == 'full' else None, blind=('test',))
        runs.append(name)
        del model
        gc.collect()
        torch.cuda.empty_cache()

In [ ]:
if MODE == 'smoke':
    ok = True
    for n in runs:
        r = log.records[n]
        first, last = r['loss_log'][0]['loss'], r['loss_log'][-1]['loss']
        v = r['metrics']['val']['ndcg']
        full_h = r['steps'] / SUBSAMPLE / r['it_per_s'] / 3600
        good = last < first and v > 0.742
        ok &= good
        print(f"{'OK   ' if good else 'CHECK'} {n:34s} steps {r['steps']:>5} | loss {first:.4f} -> {last:.4f} | "
              f"val nDCG {v} | full epoch ~{full_h:.2f} h")
    print('Smoke test passed: switch MODE to full.' if ok else
          'Some runs did not learn (loss up or val at random level): send me this output.')

## 3. Results (val)

`paired_vs`: val nDCG difference with the MSE run of the same setup and seed over the same queries; a CI that excludes 0 means a significant difference. E-over-S (share of correctly ordered E–S pairs in a query) shows whether a loss separates Exact from Substitute better, the hardest distinction in ESCI. With 5% of train (smoke) the numbers only show that training works; ce4 also starts from a random head there, so it is expected to trail.

In [ ]:
display(log.table(('ndcg', 'ndcg_ci95', 'ndcg10', 'mrr_e', 'e_over_s')))
log.print_paired()

curves = pd.DataFrame([{'run': n, **c} for n in runs for c in log.records[n]['val_curve']])
if len(curves):
    display(curves.pivot(index='step', columns='run', values='val_ndcg'))